# Liquid Financing — AI/ML Modeler Take-Home
## Barclays · Institutional-Grade Reference Implementations (Python 3.13)

This notebook drives the production modules in `src/liquid_financing/` — each module is written to
Google Python Style Guide standards (module/class/method docstrings, type hints, dataclasses) and is
independently unit-testable and importable outside the notebook. All charts are rendered with
**Plotly** and persisted to both interactive HTML (`charts/*.html`) and static PNG (`charts/*.png`,
embedded in `dissertation.pdf`).

| # | Case Study | Module |
|---|---|---|
| P1 | Sec-Lending Fee Forecasting (Elastic Net + Quantile-LightGBM + Conformal Calibration) | `p1_fee_forecasting.py` |
| P2 | Client Margin & Haircut Optimization (LASSO + GBM add-on + Kupiec/Christoffersen) | `p2_margin_optimization.py` |
| P3 | Funding-Spread Anomaly Detection (Robust Mahalanobis + Isolation Forest + TF-IDF classifier) | `p3_anomaly_detection.py` |
| P4 | Prime Balance Forecasting (Seasonal-Naive / SARIMAX / LightGBM / GRU quantile seq2seq) | `p4_balance_forecasting.py` |
| P5 | RAG Financing-Desk Copilot (BM25 + TF-IDF hybrid retrieval, reciprocal-rank fusion) | `p5_rag_copilot.py` |


In [1]:
from __future__ import annotations

import sys
sys.path.insert(0, "src")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import torch

from liquid_financing import (
    p1_fee_forecasting as p1,
    p2_margin_optimization as p2,
    p3_anomaly_detection as p3,
    p4_balance_forecasting as p4,
    p5_rag_copilot as p5,
)
from liquid_financing.plotting import apply_house_style, save_figure

np.random.seed(7)
DATA_DIR = "data"
print("Modules loaded.")


Modules loaded.


---
## P1 · Securities-Lending Fee & Rebate-Rate Forecasting

In [2]:
panel = pd.read_csv(f"{DATA_DIR}/sec_lending_panel.csv", parse_dates=["date"])
panel = panel.rename(columns={"ticker": "name_id", "gc_rate": "gc_spread"})
panel["gc_spread"] = panel["gc_spread"] - 5.0

fold_results = p1.walk_forward_backtest(panel, horizon=1, n_folds=4)
summary = pd.DataFrame([{
    "fold": i, "weighted_MAPE_%": r.weighted_mape, "directional_hit_rate": r.directional_hit_rate,
    "pinball_p10": r.pinball_loss_p10, "pinball_p90": r.pinball_loss_p90, "coverage_80%": r.coverage_80,
} for i, r in enumerate(fold_results)])
summary


,fold,weighted_MAPE_%,directional_hit_rate,pinball_p10,pinball_p90,coverage_80%
0,0,20.055731,0.875000,0.353662,0.753869,0.787031
1,1,19.986179,1.000000,0.335527,0.687447,0.799844
2,2,19.815551,1.000000,0.322983,0.680458,0.802188
3,3,19.289985,0.666667,0.339069,0.671303,0.797727


In [ ]:
# Chart 1: Fee forecast with conformalized 80% interval for one representative name
last_fold = fold_results[-1]
sample_name = last_fold.name_id.iloc[0]
mask = (last_fold.name_id == sample_name)

fig = go.Figure()
fig.add_trace(go.Scatter(x=last_fold.dates[mask], y=last_fold.y_true[mask], mode="lines+markers",
                          name="Realized fee (bps)", line=dict(color="#0B3D91", width=2)))
fig.add_trace(go.Scatter(x=last_fold.dates[mask], y=last_fold.y_pred_blend[mask], mode="lines",
                          name="Blended forecast", line=dict(color="#D62246", dash="dash")))
fig.add_trace(go.Scatter(
    x=pd.concat([last_fold.dates[mask], last_fold.dates[mask][::-1]]),
    y=np.concatenate([last_fold.y_pred_p90[mask], last_fold.y_pred_p10[mask][::-1]]),
    fill="toself", fillcolor="rgba(214,34,70,0.15)", line=dict(color="rgba(0,0,0,0)"),
    name="Conformalized 80% interval", showlegend=True,
))
apply_house_style(fig, f"P1 — Fee Forecast with Conformalized 80% Interval ({sample_name})")
fig.update_yaxes(title="Specialness fee (bps over GC)")
save_figure(fig, "p1_fee_forecast_interval", "charts")
fig.show()


In [ ]:
# Chart 2: Walk-forward weighted MAPE and coverage across folds — production monitoring view
fig = go.Figure()
fig.add_trace(go.Bar(x=summary["fold"], y=summary["weighted_MAPE_%"], name="Weighted MAPE (%)",
                      marker_color="#0B3D91", yaxis="y1"))
fig.add_trace(go.Scatter(x=summary["fold"], y=summary["coverage_80%"] * 100, name="80% interval coverage (%)",
                          mode="lines+markers", line=dict(color="#E8A33D", width=3), yaxis="y2"))
fig.add_hline(y=80, line_dash="dot", line_color="#E8A33D", annotation_text="Target coverage = 80%", yref="y2")
fig.update_layout(yaxis=dict(title="Weighted MAPE (%)"),
                   yaxis2=dict(title="Coverage (%)", overlaying="y", side="right", range=[60, 100]))
apply_house_style(fig, "P1 — Walk-Forward Backtest: Accuracy & Interval Calibration by Fold")
save_figure(fig, "p1_walkforward_monitoring", "charts")
fig.show()


---
## P2 · Client Margin & Haircut Optimization

In [ ]:
collateral = pd.read_csv(f"{DATA_DIR}/collateral_universe.csv").rename(
    columns={"volatility": "vol", "historical_haircut": "haircut"})
haircut_model, feature_cols = p2.fit_haircut_model(collateral)

coef_df = pd.DataFrame({"feature": feature_cols, "coefficient": haircut_model.coef_}).sort_values("coefficient")
fig = px.bar(coef_df, x="coefficient", y="feature", orientation="h",
             color="coefficient", color_continuous_scale=["#D62246", "#0B3D91"])
apply_house_style(fig, "P2 — LASSO Haircut Model: Feature Coefficients (log-haircut scale)", height=380)
save_figure(fig, "p2_lasso_coefficients", "charts")
fig.show()


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:1663: FutureWarning: 'n_alphas' was deprecated in 1.7 and will be removed in 1.9. 'alphas' now accepts an integer value which removes the need to pass 'n_alphas'. The default value of 'alphas' will change from None to 100 in 1.9. Pass an explicit value to 'alphas' and leave 'n_alphas' to its default value to silence this warning.
  warnings.warn(


In [ ]:
# Simulate a realistic daily P&L / breach series and run the regulatory backtest suite
rng = np.random.default_rng(11)
n_days = 250
pnl = rng.standard_t(df=5, size=n_days) * 1.2  # fat-tailed daily P&L (bps of portfolio)
var99 = np.full(n_days, 2.9)  # static 99% VaR estimate for illustration
breaches = pnl < -var99

backtest = p2.run_backtest(breaches, target_rate=0.01)
print(backtest)

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(n_days), y=pnl, mode="lines", name="Daily P&L (bps)",
                          line=dict(color="#0B3D91", width=1.3)))
fig.add_trace(go.Scatter(x=np.arange(n_days), y=-var99, mode="lines", name="99% VaR limit",
                          line=dict(color="#D62246", dash="dash")))
breach_idx = np.where(breaches)[0]
fig.add_trace(go.Scatter(x=breach_idx, y=pnl[breach_idx], mode="markers", name="VaR breach",
                          marker=dict(color="#D62246", size=10, symbol="x")))
apply_house_style(fig, f"P2 — VaR Backtest ({backtest.traffic_light} zone, "
                        f"Kupiec p={backtest.kupiec_p_value:.3f}, Christoffersen p={backtest.christoffersen_p_value:.3f})")
fig.update_yaxes(title="P&L / VaR (bps)")
save_figure(fig, "p2_var_backtest", "charts")
fig.show()


BacktestResult(n_obs=250, n_breaches=5, breach_rate=0.02, target_rate=0.01, kupiec_lr_stat=1.956809788230622, kupiec_p_value=0.1618549171960425, christoffersen_lr_stat=0.20493237552149424, christoffersen_p_value=0.6507686886878674, traffic_light='AMBER')


---
## P3 · Cross-Asset Funding-Spread Anomaly Detection

In [ ]:
spreads = pd.read_csv(f"{DATA_DIR}/funding_spreads.csv", parse_dates=["date"])
feature_cols = ["gc_sofr_spread", "sec_lending_fee_index", "xccy_basis_3m", "cdx_cash_basis"]
train, test = spreads.iloc[:400], spreads.iloc[400:].reset_index(drop=True)

monitor = p3.FundingSpreadMonitor().fit(train[feature_cols])
scored = []
for i, row in test.iterrows():
    s = monitor.score(row[feature_cols].to_numpy())
    fused = p3.fuse_scores(s["mahalanobis_z"], text_stress_prob=0.3)
    scored.append({"date": row["date"], **s, "fused_score": fused, "is_labeled_anomaly": row["is_labeled_anomaly"]})
scored_df = pd.DataFrame(scored)

fig = go.Figure()
fig.add_trace(go.Scatter(x=scored_df["date"], y=scored_df["mahalanobis_z"], mode="lines",
                          name="Robust Mahalanobis z", line=dict(color="#0B3D91")))
anomalies = scored_df[scored_df["is_labeled_anomaly"] == 1]
fig.add_trace(go.Scatter(x=anomalies["date"], y=anomalies["mahalanobis_z"], mode="markers",
                          name="True anomaly day", marker=dict(color="#D62246", size=11, symbol="star")))
apply_house_style(fig, "P3 — Robust Mahalanobis Anomaly Score on Held-Out Funding-Spread Complex")
fig.update_yaxes(title="Mahalanobis distance (z)")
save_figure(fig, "p3_anomaly_score_timeline", "charts")
fig.show()

top10 = scored_df.sort_values("fused_score", ascending=False).head(10)
top10[["date", "mahalanobis_z", "isolation_score", "fused_score", "driving_spread", "is_labeled_anomaly"]]


,date,mahalanobis_z,isolation_score,fused_score,driving_spread,is_labeled_anomaly
20,2026-01-12,25.841821,0.806963,0.720000,sec_lending_fee_index,1
66,2026-03-17,4.452195,0.533145,0.606202,xccy_basis_3m,0
2,2025-12-17,3.338743,0.507002,0.470331,cdx_cash_basis,0
75,2026-03-30,3.325577,0.521835,0.468410,gc_sofr_spread,0
8,2025-12-25,3.208595,0.533761,0.451176,cdx_cash_basis,0
78,2026-04-02,3.090739,0.536468,0.433602,sec_lending_fee_index,0
30,2026-01-26,3.018134,0.519278,0.422720,gc_sofr_spread,0
3,2025-12-18,2.939218,0.510468,0.410885,sec_lending_fee_index,0
57,2026-03-04,2.865286,0.479029,0.399823,gc_sofr_spread,0
0,2025-12-15,2.801710,0.487846,0.390354,sec_lending_fee_index,0


---
## P4 · Prime Balance & Utilization Forecasting (Deep Learning)

In [ ]:
bal_df = pd.read_csv(f"{DATA_DIR}/prime_balances.csv", parse_dates=["date"])
balance = bal_df["aggregate_balance_usd_mm"].to_numpy(dtype=np.float32)
calendar = (bal_df["calendar_flag"] == "quarter_end_window").astype(int).to_numpy()

horizon = 10
split = len(balance) - horizon - 5
history, future_actual = balance[:split], balance[split:split + horizon]
cal_hist, cal_future = calendar[:split], calendar[split:split + horizon]
future_dates = bal_df["date"].iloc[split:split + horizon]

naive = p4.seasonal_naive_forecast(history, horizon)
sarimax_fc = p4.sarimax_forecast(history, cal_hist, cal_future, horizon)
gbm_fc = p4.lightgbm_forecast(history, cal_hist, horizon)
gru_model, loss_hist, mean, std = p4.train_gru(balance[:split], calendar[:split], horizon=horizon, n_epochs=300)

with torch.no_grad():
    x_last = torch.tensor((history[-60:] - mean) / std, dtype=torch.float32).view(1, 60, 1)
    cal_last = torch.tensor(cal_future, dtype=torch.long).view(1, horizon)
    gru_out = gru_model(x_last, cal_last)
    gru_p10 = gru_out["p10"].numpy().flatten() * std + mean
    gru_p50 = gru_out["p50"].numpy().flatten() * std + mean
    gru_p90 = gru_out["p90"].numpy().flatten() * std + mean

baseline_scores = []
for name, pred in [("seasonal_naive", naive), ("sarimax", sarimax_fc), ("lightgbm", gbm_fc), ("gru_p50", gru_p50)]:
    mape = float(np.mean(np.abs((future_actual - pred) / future_actual)) * 100)
    baseline_scores.append({"model": name, "MAPE_%": mape})
baseline_df = pd.DataFrame(baseline_scores).sort_values("MAPE_%")
baseline_df


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,model,MAPE_%
0,seasonal_naive,1.356427
1,sarimax,1.442517
3,gru_p50,2.083407
2,lightgbm,2.804306


In [ ]:
# Chart: model comparison bar + GRU quantile fan around the quarter-end window
fig = px.bar(baseline_df, x="model", y="MAPE_%", color="MAPE_%", color_continuous_scale=["#2E8B57", "#D62246"])
apply_house_style(fig, "P4 — Baseline Discipline: Walk-Forward MAPE by Model", height=420)
save_figure(fig, "p4_baseline_comparison", "charts")
fig.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=list(future_dates), y=list(future_actual), mode="lines+markers",
                           name="Realized balance", line=dict(color="#0B3D91", width=2)))
fig2.add_trace(go.Scatter(x=list(future_dates), y=list(gru_p50), mode="lines", name="GRU P50",
                           line=dict(color="#D62246", dash="dash")))
fig2.add_trace(go.Scatter(
    x=list(future_dates) + list(future_dates)[::-1],
    y=list(gru_p90) + list(gru_p10)[::-1],
    fill="toself", fillcolor="rgba(214,34,70,0.15)", line=dict(color="rgba(0,0,0,0)"), name="GRU P10-P90 band",
))
qe_dates = future_dates[cal_future == 1]
for d in qe_dates:
    fig2.add_vline(x=d, line_dash="dot", line_color="#E8A33D")
apply_house_style(fig2, "P4 — GRU Quantile Forecast Around the Quarter-End Window (dotted = quarter-end)")
fig2.update_yaxes(title="Aggregate balance (USD mm)")
save_figure(fig2, "p4_gru_quantile_fan", "charts")
fig2.show()


---
## P5 · RAG Financing-Desk Copilot (Hybrid Retrieval + Grounded Synthesis)

In [ ]:
import os

def load_corpus(data_dir):
    docs = []
    policy_dir, notes_dir = f"{data_dir}/policy_docs", f"{data_dir}/desk_notes"
    if os.path.isdir(policy_dir):
        for fname in sorted(os.listdir(policy_dir)):
            text = open(os.path.join(policy_dir, fname)).read().strip()
            for i, para in enumerate([p for p in text.split(chr(10)+chr(10)) if p.strip()]):
                docs.append(p5.Document(f"{fname}#{i}", para.strip().replace(chr(10), " "), fname))
    if os.path.isdir(notes_dir):
        for fname in sorted(os.listdir(notes_dir)):
            docs.append(p5.Document(fname, open(os.path.join(notes_dir, fname)).read().strip(), fname))
    return docs

corpus = load_corpus(DATA_DIR)
retriever = p5.HybridRetriever(corpus)
print(f"Indexed {len(corpus)} corpus chunks.")

labeled_queries = [
    ("What is the BBB corporate bond haircut?", {d.doc_id for d in corpus if "BBB" in d.text}),
    ("Why did XYZ Corp fee spike?", {d.doc_id for d in corpus if "XYZ" in d.text}),
    ("What happened to the GC-SOFR spread on quarter-end?", {d.doc_id for d in corpus if "GC-SOFR" in d.text}),
    ("What is the override authority for haircuts?", {d.doc_id for d in corpus if "Override" in d.text or "override" in d.text}),
]
r_at_k = p5.recall_at_k(retriever, labeled_queries, k=5)
print(f"Recall@5 on labeled query set: {r_at_k:.2%}")

query = "Why did the XYZ Corp borrow fee spike, and what is our BBB financials haircut policy?"
retrieved = retriever.retrieve(query, top_k=4)
sim = retriever.max_dense_similarity(query)
answer = p5.generate_grounded_answer(query, retrieved, max_dense_similarity=sim)
faithfulness = p5.faithfulness_proxy(answer, [d.text for d, _ in retrieved])
print("\nCitations:", answer.citations)
print("Faithfulness proxy:", round(faithfulness, 3))

# Out-of-scope refusal demo
for oos_query in [
    "What is the weather forecast for London tomorrow?",
    "What is the CEO's view on the Q3 earnings call?",
]:
    oos_retrieved = retriever.retrieve(oos_query, top_k=4)
    oos_sim = retriever.max_dense_similarity(oos_query)
    oos_answer = p5.generate_grounded_answer(oos_query, oos_retrieved, max_dense_similarity=oos_sim)
    print(f"\nQuery: {oos_query}\n  max_dense_similarity={oos_sim:.3f}  refused={oos_answer.is_refusal}\n  -> {oos_answer.answer[:90]}")


Indexed 17 corpus chunks.
Recall@5 on labeled query set: 100.00%

Citations: ['desk_note_20260630.txt', 'haircut_policy_v7.txt#0', 'haircut_policy_v7.txt#6', 'financing_rate_policy_v4.txt#2']
Faithfulness proxy: 0.958

Query: What is the weather forecast for London tomorrow?
  max_dense_similarity=0.000  refused=True
  -> INSUFFICIENT_EVIDENCE: no retrieved document met the relevance bar for this query.

Query: What is the CEO's view on the Q3 earnings call?
  max_dense_similarity=0.000  refused=True
  -> INSUFFICIENT_EVIDENCE: no retrieved document met the relevance bar for this query.


In [ ]:
fig = go.Figure(go.Bar(
    x=[s for _, s in retrieved], y=[f"{d.doc_id}\n({d.source})" for d, _ in retrieved],
    orientation="h", marker_color="#0B3D91"))
apply_house_style(fig, "P5 — Hybrid Retrieval (BM25 + TF-IDF, RRF-fused) Scores for Sample Query", height=380)
fig.update_xaxes(title="Reciprocal Rank Fusion score")
save_figure(fig, "p5_retrieval_scores", "charts")
fig.show()


---
## Summary

All five systems are implemented as importable, independently-testable Python 3.13 modules under
`src/liquid_financing/`, each following the Google Python Style Guide (module/class/method
docstrings, explicit type hints, dataclass return types). Every chart above is persisted to
`charts/*.html` (interactive) and `charts/*.png` (embedded in `dissertation.pdf`). See `README.md`
for environment setup and `dissertation.pdf` for the full mathematical treatment.
